In [ ]:
# Simplified PacWave-style representative wave conditions
# for WEC-Sim optimization
# saves .mat files to <project_root>/wave_conditions


from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.mixture import GaussianMixture
from scipy.io import savemat

from mhkit.wave import resource
from mhkit.wave.io import ndbc
# Project folder setup
#Change the PROJECT Root for your folder location
PROJECT_ROOT = Path.home() / "WEC-Sim" / "OSWEC_Optimization_Damping" #CHANGE THIS TO YOUR LOCATION
# Change Python/Jupyter working directory to project root
os.chdir(PROJECT_ROOT)

# Save wave-condition files inside the project folder
output_folder = PROJECT_ROOT / "wave_conditions"
output_folder.mkdir(parents=True, exist_ok=True)

print("--------------------------------------------")
print("Current working directory:")
print(Path.cwd())
print("--------------------------------------------")
print(f"Project root: {PROJECT_ROOT}")
print(f"Output folder: {output_folder}")
print("--------------------------------------------")

# INPUTS
# NDBC buoy number
buoy_number = "46050"

# Years to analyze
# Example: ["2020"] or ["2019", "2020", "2021"]
years = ["2020", "2021", "2022", "2023", "2024", "2025"]

# Number of representative wave conditions to create

clusters = [4, 8, 16, 32, 64]

# Water depth in meters
# Use the buoy/site water depth you want for energy flux calculation
#(see MHKIT example for more details on this)
water_depth = 160.0

# Choose which cluster result to export to .mat for WEC-Sim
# Set to a number like 32 to export only one file
# Set to None to export all clusters
cluster_to_export = None
# Download spectral wave density data from NDBC

parameter = "swden"

print(f"Buoy number: {buoy_number}")
print(f"Years: {years}")
print(f"Clusters: {clusters}")
print(f"Water depth: {water_depth} m")
print("--------------------------------------------")

print("Checking available NDBC data...")

ndbc_available_data = ndbc.available_data(parameter, buoy_number)

# Create a clean year string for filenames
if years is None:
    years_string = "all_years"
else:
    years = [str(year) for year in years]
    years_string = "_".join(years)

    # Filter available files by selected years
    year_pattern = "|".join(years)

    ndbc_available_data = ndbc_available_data[
        ndbc_available_data["filename"].astype(str).str.contains(year_pattern)
    ]

    if ndbc_available_data.empty:
        raise ValueError(f"No data files found for buoy {buoy_number} and years {years}.")

filenames = ndbc_available_data["filename"]

print("Downloading NDBC data...")

ndbc_requested_data = ndbc.request_data(parameter, filenames)

# Clean NDBC data and create DateTime index


ndbc_data = {}

for year in ndbc_requested_data:
    print(f"Cleaning data for year {year}...")

    data_raw = ndbc_requested_data[year].copy()

    # Create datetime index from NDBC date/time columns
    data_raw["date"] = pd.to_datetime(
        {
            "year": data_raw["#YY"],
            "month": data_raw["MM"],
            "day": data_raw["DD"],
            "hour": data_raw["hh"],
            "minute": data_raw["mm"],
        },
        errors="coerce",
    )

    data_raw = data_raw.set_index("date")

    # Drop original date/time columns
    data_raw = data_raw.drop(columns=["#YY", "MM", "DD", "hh", "mm"])

    # Convert frequency column names to floats
    new_columns = []

    for col in data_raw.columns:
        col_string = str(col)

        if col_string.startswith("."):
            col_string = "0" + col_string

        new_columns.append(float(col_string))

    data_raw.columns = new_columns

    # Convert data values to numeric
    data_raw = data_raw.apply(pd.to_numeric, errors="coerce")

    # Replace NDBC missing/bad values
    data_raw = data_raw.replace([999.0, 99.0], np.nan)

    # Drop rows with missing data
    data_raw = data_raw.dropna()

    # Sort by time
    data_raw = data_raw.sort_index()

    ndbc_data[str(year)] = data_raw

print("Cleaned data years:")
print(list(ndbc_data.keys()))

# Calculate QoIs from this data


Hm0_list = []
Te_list = []
J_list = []
Tp_list = []
Tz_list = []

for year in ndbc_data:
    print(f"Calculating QoIs for year {year}...")

    data_raw = ndbc_data[year]

    year_data = data_raw[data_raw != 999.0].dropna()

    # MHKiT expects frequency as rows and timestamps as columns
    spectrum = year_data.T

    Hm0_list.append(resource.significant_wave_height(spectrum))
    Te_list.append(resource.energy_period(spectrum))
    J_list.append(resource.energy_flux(spectrum, h=water_depth))

    # Peak period calculation from PacWave example
    fp = spectrum.idxmax(axis=0).astype(float)
    Tp = 1 / fp
    Tp = pd.DataFrame(Tp, index=spectrum.columns, columns=["Tp"])
    Tp_list.append(Tp)

    Tz_list.append(resource.average_zero_crossing_period(spectrum))


Te = pd.concat(Te_list, axis=0)
Tp = pd.concat(Tp_list, axis=0)
Hm0 = pd.concat(Hm0_list, axis=0)
J = pd.concat(J_list, axis=0)
Tz = pd.concat(Tz_list, axis=0)

# Name each Series/DataFrame
Te.name = "Te"
Tp.name = "Tp"
Hm0.name = "Hm0"
J.name = "J"
Tz.name = "Tz"

# Combine into one DataFrame
data = pd.concat([Hm0, Te, Tp, J, Tz], axis=1)

# Make sure columns are named correctly
data.columns = ["Hm0", "Te", "Tp", "J", "Tz"]

# Calculate wave steepness
data["Sm"] = data.Hm0 / (9.81 / (2 * np.pi) * data.Tz**2)

# Drop NaNs and sort
data.dropna(inplace=True)
data.sort_index(inplace=True)

print("--------------------------------------------")
print("QoI data preview:")
print(data.head())
print("--------------------------------------------")


data_clean = data[data.Hm0 < 20]

sigma = data_clean.J.std()
data_clean = data_clean[data_clean.J > (data_clean.J.mean() - 0.9 * sigma)]

print(f"Number of sea states after cleaning: {len(data_clean)}")



# PacWave-style clustering using Gaussian Mixture model


X = np.vstack((data_clean.Te.values, data_clean.Hm0.values)).T

fig, axs = plt.subplots(len(clusters), 1, figsize=(8, 5 * len(clusters)), sharex=True)

if len(clusters) == 1:
    axs = [axs]

results = {}

for cluster in clusters:
    gmm = GaussianMixture(n_components=cluster).fit(X)

    # Save centers and weights
    result = pd.DataFrame(gmm.means_, columns=["Te", "Hm0"])
    result["weights"] = gmm.weights_

    # Same relationship used in the PacWave example
    result["Tp"] = result.Te / 0.858

    results[cluster] = result

    labels = gmm.predict(X)

    i = clusters.index(cluster)

    axs[i].scatter(
        data_clean.Te.values,
        data_clean.Hm0.values,
        c=labels,
        s=40,
    )

    axs[i].plot(
        result.Te,
        result.Hm0,
        "m+",
    )

    axs[i].title.set_text(f"{cluster} Clusters")
    plt.setp(axs[i], ylabel="Sig. wave height, $Hm0$ [m]")

plt.setp(axs[len(clusters) - 1], xlabel="Energy Period, $T_e$ [s]")

plt.tight_layout()
plt.show()



# View the representative wave conditions


if cluster_to_export is not None:
    print("--------------------------------------------")
    print(f"Representative wave conditions for {cluster_to_export} clusters:")
    print("--------------------------------------------")
    print(results[cluster_to_export])



# Save representative wave conditions to .mat for WEC-Sim


# Decide which cluster counts to export
if cluster_to_export is None:
    clusters_to_save = clusters
else:
    clusters_to_save = [cluster_to_export]

for cluster in clusters_to_save:
    result = results[cluster]

    wave_conditions = {
        "Hm0": result["Hm0"].to_numpy(),
        "Te": result["Te"].to_numpy(),
        "Tp": result["Tp"].to_numpy(),
        "weights": result["weights"].to_numpy(),
        "cluster_to_export": np.array([cluster]),
        "buoy_number": buoy_number,
        "years_string": years_string,
    }

    output_mat_file = output_folder / (
        f"wave_conditions_buoy_{buoy_number}_{years_string}_{cluster}_clusters.mat"
    )

    savemat(str(output_mat_file), wave_conditions)

    print("--------------------------------------------")
    print("Saved WEC-Sim wave condition file:")
    print(output_mat_file)
    print("--------------------------------------------")